# Procesamiento ETL 2024 - Estadísticas Vitales

Este notebook se encarga de importar, limpiar, manejar nulos, normalizar y guardar de forma eficiente (en formato Parquet) los datasets correspondientes al año 2024 para:
- Nacimientos
- Muertes Fetales
- Muertes No Fetales

In [1]:
import os
import sys
import pandas as pd

# Agregar la carpeta principal al path para poder importar módulos propios
sys.path.append(os.path.abspath('..'))

from etl.cleaning import *
from etl.nulls import *
from etl.normalization import *
from etl.schema import *

# Asegurarnos de que directorios de salida existan
os.makedirs('../data/processed/nacimientos', exist_ok=True)
os.makedirs('../data/processed/fetales', exist_ok=True)
os.makedirs('../data/processed/no_fetales', exist_ok=True)

## 1. Nacimientos 2024

### Carga del archivo

In [2]:
rutaDatos = '../data/raw/nac2024.csv'
try:
    df_nac = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_nac.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 453901 entries, 0 to 453900
Data columns (total 40 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   COD_DPTO        453901 non-null  int64  
 1   COD_MUNIC       453901 non-null  int64  
 2   AREANAC         453901 non-null  int64  
 3   SIT_PARTO       453901 non-null  int64  
 4   OTRO_SIT        980 non-null     object 
 5   SEXO            453901 non-null  int64  
 6   PESO_NAC        453901 non-null  int64  
 7   TALLA_NAC       453901 non-null  int64  
 8   ANO             453901 non-null  int64  
 9   MES             453901 non-null  int64  
 10  ATEN_PAR        453901 non-null  int64  
 11  T_GES           453901 non-null  int64  
 12  T_GES_AGRU_CIE  453901 non-null  int64  
 13  NUMCONSUL       453901 non-null  int64  
 14  TIPO_PARTO      453901 non-null  int64  
 15  MUL_PARTO       453901 non-null  int64  
 16  APGAR1          453901 non-null  i

### Eliminación de duplicados

In [3]:
encontrar_duplicados(df_nac)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 453901
Filas duplicadas (incluyendo repetidas): 4666
Duplicados únicos a eliminar: 2906


,COD_DPTO,COD_MUNIC,AREANAC,SIT_PARTO,OTRO_SIT,SEXO,PESO_NAC,TALLA_NAC,ANO,MES,...,N_HIJOSV,FECHA_NACM,N_EMB,SEG_SOCIAL,IDCLASADMI,EDAD_PADRE,NIV_EDUP,ULTCURPAD,PROFESION,TIPOFORMULARIO
1370,54,1,1,1,NaN,1,4,4,2024,1,...,2,NaN,1,2,2.0,27,3,9,1.0,1
1388,8,1,1,1,NaN,2,4,4,2024,4,...,4,27/09/2007,4,1,1.0,36,9,4,1.0,1
1960,8,1,1,1,NaN,2,4,4,2024,4,...,4,27/09/2007,4,1,1.0,36,9,4,1.0,1
2094,5,615,1,1,NaN,1,4,4,2024,11,...,2,23/03/2018,3,2,2.0,20,4,10,1.0,1
2862,70,1,1,1,NaN,1,4,4,2024,1,...,2,NaN,1,2,2.0,30,7,2,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450659,11,1,1,1,NaN,1,4,4,2024,11,...,1,NaN,2,1,1.0,24,4,11,1.0,1
451637,54,1,1,1,NaN,1,5,4,2024,1,...,4,18/09/2022,4,1,1.0,42,9,5,1.0,1
451638,54,1,1,1,NaN,1,5,4,2024,1,...,4,18/09/2022,4,1,1.0,42,9,5,1.0,1
453079,13,1,1,1,NaN,1,5,4,2024,3,...,4,02/01/2021,4,1,1.0,33,7,2,1.0,1


In [4]:
df_nac = eliminar_duplicados(df_nac)

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 2906
Registros eliminados: 2906
Total final: 450995


### Eliminación de columnas que no son necesarias para el estudio

In [5]:
cols_a_borrar_nac = ['CODPAISNACMAD','IDPERTET','OTRO_SIT','APGAR1','APGAR2','IDPERTE','ULTCURMAD','H_HIJOSV','FECHA_NACM','N_EMB','ULTCURPAD']
df_nac = eliminar_columnas(df_nac, cols_a_borrar_nac)

Borradas 9 columnas de la lista especificada.


### Estandarización de Nombres de las columnas

In [6]:
# 3. Estandarizar Nombres de Columnas
df_nac = estandarizar_nombres_columnas(df_nac)

### Nueva columna

In [7]:
df_nac["tipo_evento"] = "NAC"

### Manejo de nulos 

In [8]:
resumen_nulos = analizar_nulos(df_nac)

===== ANÁLISIS DE NULOS =====
Total de registros: 450995
Columnas con nulos: 6



,nulos,porcentaje (%)
idclasadmi,16164,3.584075
area_res,8219,1.822415
codmunre,8219,1.822415
codptore,8219,1.822415
profesion,6056,1.342809
codpres,5772,1.279837


Al realizar el análisis de valores nulos, se identificó que las variables codptore, codmunre y area_res presentan una alta correlación en sus valores faltantes, es decir, los registros con valores nulos coinciden en las mismas filas. Esto indica la existencia de datos incompletos en el componente geográfico, por lo que se decidió eliminar dichos registros para garantizar la consistencia del análisis territorial.

Por otro lado, para la variable idclasadmi, se definió una estrategia de imputación en lugar de eliminación, debido a que presenta un porcentaje significativo de valores nulos (~11.46%). Eliminar estos registros implicaría una pérdida considerable de información. Además, al tratarse de una variable categórica asociada al régimen de salud, su ausencia no impide el análisis principal, por lo que se optó por reemplazar los valores faltantes con la categoría “DESCONOCIDO”, preservando así la integridad del dataset.

In [9]:
df_nac[df_nac["codptore"].isnull()].shape


(8219, 32)

In [10]:
df_nac[df_nac["codmunre"].isnull()].shape

(8219, 32)

In [11]:
df_nac[df_nac["codmunre"].isnull()].shape



(8219, 32)

In [12]:
df_nac[df_nac["area_res"].isnull()].shape

(8219, 32)

In [13]:
# 2. eliminar nulos geográficos
df_nac = df_nac.dropna(subset=["codptore", "codmunre", "area_res"])

In [14]:
# 3. imputar otros nulos
estrategia = {
    "idclasadmi": {"metodo": "fill", "valor": "DESCONOCIDO"},
    "profesion": {"metodo": "fill", "valor": "DESCONOCIDO"}
    
}

df_nac = manejar_nulos(df_nac, estrategia)

In [15]:
resumen_nulos = analizar_nulos(df_nac)

===== ANÁLISIS DE NULOS =====
Total de registros: 442776
Columnas con nulos: 0



,nulos,porcentaje (%)


In [16]:
df_nac['profesion'].value_counts()

profesion
1              440933
3                 666
5                 507
DESCONOCIDO       286
4                 221
2                 163
Name: count, dtype: int64

### Ajustar Tipos de Datos (Bajar peso en memoria)

In [17]:
df_nac = ajustar_tipos_datos(df_nac)

### Verficación de columnas

In [18]:
VALID_COLUMNS_NACIMIENTOS = {
    col: str(df_nac[col].dtype) for col in VALID_COLUMNS_NACIMIENTOS
}

validar_schema(df_nac, VALID_COLUMNS_NACIMIENTOS)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 30

➕ Columnas extra (2): ['t_ges_agru_cie', 'tipoformulario']


In [19]:
cols_a_borrar_nac_extra = ['t_ges_agru_cie', 'tipoformulario']
df_nac = eliminar_columnas(df_nac, cols_a_borrar_nac_extra)

Borradas 2 columnas de la lista especificada.


In [20]:
validar_schema(df_nac, VALID_COLUMNS_NACIMIENTOS)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 30


## 2. Muertes Fetales 2024

### Carga de dataset

In [21]:
rutaDatos = '../data/raw/fetal2024.csv'
try:
    df_fet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_fet.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21250 entries, 0 to 21249
Data columns (total 46 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   COD_DPTO        21250 non-null  int64  
 1   COD_MUNIC       21250 non-null  int64  
 2   A_DEFUN         21250 non-null  int64  
 3   SIT_DEFUN       21250 non-null  int64  
 4   OTRSITIODE      36 non-null     object 
 5   TIPO_DEFUN      21250 non-null  int64  
 6   ANO             21250 non-null  int64  
 7   MES             21250 non-null  int64  
 8   HORA            9791 non-null   float64
 9   MINUTOS         9791 non-null   float64
 10  SEXO            21250 non-null  int64  
 11  CODPRES         21157 non-null  float64
 12  CODPTORE        20981 non-null  float64
 13  CODMUNRE        20981 non-null  float64
 14  AREA_RES        20982 non-null  float64
 15  P_PMAN_IRIS     21250 non-null  int64  
 16  CONS_EXP        21250 non-null  int64  
 17  MU_PARTO 

### Eliminación de duplicados

En el conjunto de datos de muertes fetales se identificaron 369 registros duplicados, lo que representa aproximadamente el 1.10% del total. Este porcentaje se considera bajo, por lo que se procedió a su eliminación sin impacto significativo en la integridad del análisis.

In [22]:
encontrar_duplicados(df_fet)

===== ANÁLISIS DE DUPLICADOS =====


Total de registros: 21250
Filas duplicadas (incluyendo repetidas): 432
Duplicados únicos a eliminar: 264


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,...,C_MUERTED,C_MUERTEE,C_MUERTEF,C_MUERTEG,ASIS_MED,CAUSA_MULT,C_BAS1,CAUSA_667,IDPROFCER,CAU_HOMOL
22,13,1,1,1,NaN,1,2024,8,NaN,NaN,...,NaN,NaN,NaN,NaN,1,P018,P018,402,1,80
37,25,269,1,1,NaN,1,2024,11,NaN,NaN,...,NaN,NaN,NaN,NaN,1,P018,P018,402,1,80
90,25,269,1,1,NaN,1,2024,11,NaN,NaN,...,NaN,NaN,NaN,NaN,1,P018,P018,402,1,80
91,25,269,1,1,NaN,1,2024,11,NaN,NaN,...,NaN,NaN,NaN,NaN,1,P018,P018,402,1,80
92,25,269,1,1,NaN,1,2024,11,NaN,NaN,...,NaN,NaN,NaN,NaN,1,P018,P018,402,1,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21160,11,1,1,1,NaN,1,2024,10,NaN,NaN,...,NaN,NaN,NaN,NaN,1,P018,P018,402,1,80
21184,13,1,1,1,NaN,1,2024,5,19.0,10.0,...,NaN,NaN,NaN,NaN,1,P964*P964,P964,406,1,86
21218,25,269,1,1,NaN,1,2024,9,NaN,NaN,...,NaN,NaN,NaN,NaN,1,P018,P018,402,1,80
21226,25,269,1,1,NaN,1,2024,9,NaN,NaN,...,NaN,NaN,NaN,NaN,1,P018,P018,402,1,80


In [23]:
df_nac = eliminar_duplicados(df_fet)

===== LIMPIEZA DE DUPLICADOS =====


Duplicados detectados: 264
Registros eliminados: 264
Total final: 20986


### Eliminación de columnas que no son necesarias para el estudio

In [24]:
cols_a_borrar_fet = ['HORA','MINUTOS','OTRSITIODE','CODPRES','P_PMAN_IRIS','CONS_EXP','ULTCURMAD','CODOCUR','CODMUNOC','C_MUERTE','C_MUERTEB','C_MUERTEC','C_MUERTED','C_MUERTEE','CAUSA_MULT','C_BAS1','CAUSA_667','IDPROFCER']
df_fet = eliminar_columnas(df_fet, cols_a_borrar_fet)

Borradas 18 columnas de la lista especificada.


### Estandarizar Nombres de Columnas

In [25]:
df_fet = estandarizar_nombres_columnas(df_fet)

### Nueva Columna 

In [26]:
df_fet["tipo_evento"] = "FETAL"

### Manejo de nulos

In [27]:
resumen_nulos = analizar_nulos(df_fet)

===== ANÁLISIS DE NULOS =====
Total de registros: 21250
Columnas con nulos: 6



,nulos,porcentaje (%)
c_muerteg,21249,99.995294
c_muertef,21228,99.896471
codptore,269,1.265882
codmunre,269,1.265882
area_res,268,1.261176
codpaisnacmad,142,0.668235


In [28]:
df_fet[df_fet["codptore"].isnull()].shape


(269, 29)

In [29]:
df_fet[df_fet["codmunre"].isnull()].shape

(269, 29)

In [30]:
df_fet[df_fet["area_res"].isnull()].shape

(268, 29)

In [31]:
df_fet = df_fet.dropna(subset=["codptore", "codmunre", "area_res"])

In [32]:
estrategia = {
    "idadmisalud": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_fet = manejar_nulos(df_fet, estrategia)

In [33]:
resumen_nulos = analizar_nulos(df_fet)

===== ANÁLISIS DE NULOS =====
Total de registros: 20981
Columnas con nulos: 3



,nulos,porcentaje (%)
c_muerteg,20981,100.000000
c_muertef,20959,99.895143
codpaisnacmad,48,0.228778


### Ajustar Tipos de Datos (Bajar peso en memoria)

In [34]:
df_fet = ajustar_tipos_datos(df_fet)

### Verficación de columnas

In [35]:
VALID_COLUMNS_FETALES = {
    col: str(df_fet[col].dtype) for col in VALID_COLUMNS_FETALES
}

validar_schema(df_fet, VALID_COLUMNS_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 24

➕ Columnas extra (5): ['t_ges_agru_cie', 'codpaisnacmad', 'regsocialmadre', 'c_muertef', 'c_muerteg']


In [36]:
cols_a_borrar_fet_extra = ['t_ges_agru_cie', 'codpaisnacmad', 'regsocialmadre', 'c_muertef', 'c_muerteg']
df_fet = eliminar_columnas(df_fet, cols_a_borrar_fet_extra)

Borradas 5 columnas de la lista especificada.


In [37]:
validar_schema(df_fet, VALID_COLUMNS_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 24


## 3. Muertes No Fetales 2024

### Carga de Dataset

In [38]:
rutaDatos = '../data/raw/nofetal2024.csv'
try:
    df_nofet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_nofet.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275778 entries, 0 to 275777
Data columns (total 61 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   COD_DPTO        275778 non-null  int64  
 1   COD_MUNIC       275778 non-null  int64  
 2   A_DEFUN         275778 non-null  int64  
 3   SIT_DEFUN       275778 non-null  int64  
 4   OTRSITIODE      7713 non-null    object 
 5   TIPO_DEFUN      275778 non-null  int64  
 6   ANO             275778 non-null  int64  
 7   MES             275778 non-null  int64  
 8   HORA            268273 non-null  float64
 9   MINUTOS         268273 non-null  float64
 10  CODPAISNACFAL   236617 non-null  float64
 11  SEXO            275778 non-null  int64  
 12  EST_CIVIL       275778 non-null  int64  
 13  GRU_ED1         275778 non-null  int64  
 14  GRU_ED2         275778 non-null  int64  
 15  NIVEL_EDU       275778 non-null  int64  
 16  ULTCURFAL       275778 non-null  i

### Eliminar duplicados

In [39]:
encontrar_duplicados(df_nofet)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 275778
Filas duplicadas (incluyendo repetidas): 194
Duplicados únicos a eliminar: 102


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,...,C_MUERTEE,C_MUERTEF,C_MUERTEG,ASIS_MED,CAUSA_MULT,C_BAS1,CAUSA_667,IDPROFCER,CAU_HOMOL,TIPOFORMULARIO
11334,11,1,1,3,NaN,2,2024,12,0.0,0.0,...,NaN,NaN,NaN,2,R570/I219/I251,I219,303,1,51,1
16438,13,1,1,5,NaN,2,2024,8,0.0,0.0,...,NaN,NaN,NaN,2,S062/S097/T141 X95,X954,512,1,101,1
23455,11,1,1,3,NaN,2,2024,11,0.0,0.0,...,NaN,NaN,NaN,2,R98/R98/R98,R98,0,1,89,1
25649,11,1,1,3,NaN,2,2024,12,0.0,0.0,...,NaN,NaN,NaN,2,R570/I219/I251,I219,303,1,51,1
37689,18,94,3,5,NaN,2,2024,6,0.0,0.0,...,NaN,NaN,NaN,2,T794/S062/T141 X93,X934,512,1,101,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228452,63,401,3,6,RIO ARROYO HUMEDAL RONDA LAGO EMBALSE,2,2024,7,0.0,0.0,...,NaN,NaN,NaN,2,T794/S097/T141 X95,X958,512,1,101,1
232795,5,79,9,9,NaN,2,2024,6,0.0,0.0,...,NaN,NaN,NaN,2,G931/S131/S199 Y34,Y349,513,1,102,1
238687,11,1,1,1,NaN,2,2024,1,0.0,0.0,...,NaN,NaN,NaN,2,G932/I639/Q204,Q204,613,1,87,1
247998,5,1,1,3,NaN,2,2024,12,0.0,0.0,...,NaN,NaN,NaN,2,R98/R98,R98,0,1,89,1


In [40]:
df_nac = eliminar_duplicados(df_nofet)

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 102
Registros eliminados: 102
Total final: 275676


### Eliminación de columnas que no nos sirven para el estudio 

In [41]:
cols_a_borrar_nofet = ['OCUPACION','HORA','MINUTOS','OTRSITIODE','CODPRES','P_PMAN_IRIS','CONS_EXP','ULTCURMAD','CODOCUR','CODMUNOC','C_MUERTE','C_MUERTEB','C_MUERTEC','C_MUERTED','C_MUERTEE','CAUSA_MULT','C_BAS1','CAUSA_667','IDPROFCER','MU_PARTO','CAU_HOMOL']
df_nofet = eliminar_columnas(df_nofet, cols_a_borrar_nofet)

Borradas 21 columnas de la lista especificada.


### Estandarizar nombre de columnas

In [42]:
df_nofet = estandarizar_nombres_columnas(df_nofet)

### Nueva columna

In [43]:
df_nofet["tipo_evento"] = "NOFETAL"

### Manejo de nulos
El conjunto de datos de muertes no fetales presenta un alto porcentaje de valores nulos en múltiples variables, superando el 90% en varios casos. Esto sugiere que dichas variables no son aplicables o no fueron diligenciadas en este tipo de registros. Por esta razón, se optó por eliminar estas columnas para evitar ruido en el análisis.

Adicionalmente, las variables geográficas presentaron un bajo porcentaje de valores nulos (~0.5%), los cuales fueron eliminados para garantizar la consistencia espacial. Finalmente, las variables con porcentajes moderados de valores faltantes fueron tratadas mediante imputación, con el fin de preservar la mayor cantidad de información posible.

In [44]:
resumen_nulos = analizar_nulos(df_nofet)

===== ANÁLISIS DE NULOS =====
Total de registros: 275778
Columnas con nulos: 23



,nulos,porcentaje (%)
c_muerteg,275725,99.980782
simuertepo,275677,99.963376
c_muertef,274958,99.702659
codpaisnacmad,271407,98.415029
t_parto,271019,98.274337
t_ges_agru_cie,271019,98.274337
t_ges,271019,98.274337
tipo_emb,271019,98.274337
n_hijosv,271019,98.274337
peso_nac,271019,98.274337


In [45]:
#  eliminar nulos geográficos
df_nofet = df_nofet.dropna(subset=["codptore", "codmunre", "area_res"])

In [46]:
estrategia = {
    "ocupacion": {"metodo": "fill", "valor": "DESCONOCIDO"},
    "muerteporo": {"metodo": "fill_mode"},
    "idadmisalud": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_nofet = manejar_nulos(df_nofet, estrategia)

In [47]:
columnas_eliminar = [
    "simuertepo",
    "t_ges",
    "niv_edum",
    "n_hijosm",
    "tipo_emb",
    "t_parto",
    "est_civm",
    "peso_nac",
    "edad_madre",
    "n_hijosv",
    "emb_mes",
    "emb_sem",
    "emb_fal"
]

In [48]:
# eliminar columnas inútiles (>90% nulos)
df_nofet = df_nofet.drop(columns=columnas_eliminar)

### Ajustar Tipos de Datos (Bajar peso en memoria)

In [49]:
df_nofet = ajustar_tipos_datos(df_nofet)

### Verficación de columnas

In [50]:
VALID_COLUMNS_NO_FETALES = {
    col: str(df_nofet[col].dtype) for col in VALID_COLUMNS_NO_FETALES
}

validar_schema(df_nofet, VALID_COLUMNS_NO_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 21

➕ Columnas extra (7): ['codpaisnacfal', 't_ges_agru_cie', 'codpaisnacmad', 'regsocialmadre', 'c_muertef', 'c_muerteg', 'tipoformulario']


In [51]:
cols_a_borrar_fet_extra = ['codpaisnacfal','t_ges_agru_cie', 'codpaisnacmad','regsocialmadre','c_muertef','c_muerteg','tipoformulario']
df_nofet = eliminar_columnas(df_nofet, cols_a_borrar_fet_extra)

Borradas 7 columnas de la lista especificada.


In [52]:
validar_schema(df_nofet, VALID_COLUMNS_NO_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 21


## Guardar datos procesados

In [53]:
AÑO = 2024

out_nac = f"../data/processed/nacimientos/nac{AÑO}.parquet"
out_fet = f"../data/processed/fetales/fet{AÑO}.parquet"
out_nofet = f"../data/processed/no_fetales/nofet{AÑO}.parquet"

df_nac.to_parquet(out_nac, index=False)
df_fet.to_parquet(out_fet, index=False)
df_nofet.to_parquet(out_nofet, index=False)

print(f"Guardado: {out_nac}")
print(f"Guardado: {out_fet}")
print(f"Guardado: {out_nofet}")

Guardado: ../data/processed/nacimientos/nac2024.parquet
Guardado: ../data/processed/fetales/fet2024.parquet
Guardado: ../data/processed/no_fetales/nofet2024.parquet
